In [1]:
%matplotlib inline
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

#  import utility functions from existing modules
from train_cvae import load_data
from ae_utils import _encode, _decode

In [2]:
root = Path.cwd().parent
dataset_type = 'scenairo'

if dataset_type == 'mnist':
    checkpoint_path = root / 'ASAB/runs/0904_mnist/vae_1.pth'
    latent_dim = 12
    data_csv = None

if dataset_type == 'scenairo':
    checkpoint_path = root / 'ASAB/runs/scenairo2004/vae_1.pth'
    latent_dim = 32
    data_csv = None

data = load_data(
    data_csv=data_csv,
    dataset=dataset_type,
    single_class=None,
    data_root=root / 'ASAB/DATA/ScenAIro'
)

X_train_tensor = data['X_train_tensor']
y_train = data['y_train']
input_dim = data['input_dim']
num_classes = data['class_size']
image_shape = data['image_shape']


Class mapping: {'norunway': 0, 'runway': 1}


In [3]:
target_class = 3 if dataset_type == 'mnist' else 1
latent_vectors = _encode(target_class,
                        num_classes,
                        input_dim,
                        latent_dim,
                        checkpoint_path,
                        dataset_type,
                        X_train_tensor,
                        y_train,
                        image_shape=image_shape)

print(latent_vectors)


tensor([[ 0.8492, -0.5023, -0.1867,  ..., -0.2138, -1.0802, -0.2834],
        [ 0.6822, -0.5164, -0.0407,  ..., -0.4647, -0.7095, -0.1405],
        [ 0.0566,  0.6102, -0.1510,  ..., -0.6241, -0.3423, -0.3740],
        ...,
        [-0.1752, -0.0307, -0.0447,  ...,  0.6765,  0.1377,  0.0451],
        [-0.0628,  1.5876, -0.0125,  ...,  0.3984, -0.6350, -0.5404],
        [-0.0881,  0.5432,  0.1066,  ...,  0.5846,  0.7596, -0.1425]])


In [4]:
class_mask = y_train == target_class
X_train_class = X_train_tensor[class_mask]

latent_vectors = _encode(target_class,
                        num_classes,
                        input_dim,
                        latent_dim,
                        checkpoint_path,
                        dataset_type,
                        X_train_tensor,
                        y_train,
                        image_shape=image_shape)

n = min(5, len(latent_vectors))
fig, axes = plt.subplots(2, n, figsize=(n*2, 4))

for i in range(n):
    idx = torch.randint(len(latent_vectors), (1,)).item()
    orig = X_train_class[idx]

    reconstructed = _decode(
        latent_vectors[idx:idx+1],
        target_class,
        num_classes,
        input_dim,
        latent_dim,
        checkpoint_path,
        dataset_type,
        image_shape=image_shape
    )

    if dataset_type == 'mnist':
        axes[0, i].imshow(orig.numpy().reshape(28, 28), cmap='gray')
        axes[1, i].imshow(reconstructed[0].reshape(28, 28), cmap='gray')
    else:
        axes[0, i].imshow(orig.permute(1, 2, 0).numpy())
        axes[1, i].imshow(reconstructed[0].permute(1, 2, 0).detach().numpy())

    axes[0, i].set_title(f'Original {i}')
    axes[1, i].set_title(f'Reconstructed {i}')
    axes[0, i].axis('off')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('reconstruction_preview.png', dpi=150)
plt.show()


/tmp/ipykernel_1340577/2038273097.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
